In [3]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

sqlite_index.count()

85

In [4]:
results = sqlite_index.search("Can I join facebook", num_results=10)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I’ve already submitted my project. Why can’t I review any projects?',
 'Can I use Bluesky for learning in public credits?',
 'Where can I find previous LLM Zoomcamp projects?',
 'Can I use the workshop materials for my own projects or share them with others?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Can I run the course locally instead of Codespaces?',
 'Is it a group project?',
 'Homework: Why does the content keep changing?']

In [5]:
import os
from dotenv import load_dotenv
from google.genai import Client
# from rag_helper import RAGBase

# Load the .env file
load_dotenv()
print(f"Key found: {os.environ.get('GEMINI_API_KEY') is not None}")

Key found: True


In [6]:
gemini_api_client = Client(api_key=os.environ.get('GEMINI_API_KEY'))
print(f"Client created: {gemini_api_client is not None}")

Client created: True


In [7]:
from google.genai import types

def search(query: str) -> str:
    """Search the Frequently Asked Questions database for entries matching the given query."""
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    results = sqlite_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )
    return results


search_function = types.FunctionDeclaration(
    name="search",
    description="Search the Frequently Asked Questions database for entries matching the query.",
    parameters_json_schema={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query"
            }
        },
        "required": ["query"]
    }
)

search_tool = types.Tool(function_declarations=[search_function])


In [14]:
print(search("Can I join facebook?"))

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': 'dbf5369006', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Can I use Bluesky for learning in public credits?', 'answer': 'Yes. Bluesky posts can be used for learning in public credits.'}, {'id': 'a8a7fef016', 'course': 'llm-zoomcamp', 'section': 'Capstone Project', 'question': 'I’ve already submitted my project. Why can’t I review any projects?', 'answer': "Once the project submission deadline has passed, projects will be assigned to you for evaluation. You can't choose which projects to evaluate, and you can’t review before the list has been released."}, {'id': '930286278d', 'course': 'llm-zoomcamp', 'section': 'Capstone Project', 'question': 'Where c

In [ ]:

INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the provided context does not contain information, use the search tool to further find it."
'''

def build_context(search_results):
    '''Builds a context string from the search results, and the prompt template
       to be used in the prompt for the LLM.'''     
    lines = []

    for doc in search_results:
      lines.append(doc['section'])
      lines.append('Q: ' + doc['question'])
      lines.append('A: ' + doc['answer'])
      lines.append('')

    return '\n'.join(lines).strip()


def build_prompt(question: str, search_results: str) -> str:
    """Builds a prompt for the LLM using the provided question and search results."""
    context = build_context(search_results)
    prompt = f"""
    Context:
    {context}

    Question:
    {question}
    """
    return prompt.strip()

In [9]:
def llm_gemini(prompt: str) -> str:
    """Uses the Gemini API Client to call the LLM and get the response."""
    response = gemini_api_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[search_tool]
        )
    )

    # Check Function Call
    print("Function call requested:", response.candidates[0].content.parts[0].function_call)
    return response.text


def ask(question: str) -> str:
    """Answers a question by searching the FAQ database and using the LLM to generate a response."""
    search_results = search(question)
    prompt = build_prompt(question, search_results)
    answer = llm_gemini(prompt)
    return answer

In [33]:
print(ask("Can I join facebook?"))

Function call requested: None
I don't know.


In [39]:
def ask_beta(question: str) -> str:
    response = gemini_api_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=question,
        config=types.GenerateContentConfig(
            tools=[search_tool],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(
                disable=True
            ),
            tool_config=types.ToolConfig(
                function_calling_config=types.FunctionCallingConfig(mode="ANY")
            ),
        ),
    )

    if response.function_calls:
        print("Tool requested:", response.function_calls[0].name)
        print("Args:", response.function_calls[0].args)
    else:
        print("No tool call requested")

    print(response.function_calls[0])
    print(type(response.function_calls[0]))

    return response.text

In [40]:
print(ask_beta("Can I join facebook?"))

Tool requested: search
Args: {'query': 'joining facebook'}
id=None args={'query': 'joining facebook'} name='search' partial_args=None will_continue=None
<class 'google.genai.types.FunctionCall'>
None


In [24]:
def llm_gemini_2(prompt: str) -> str:
    """Uses the Gemini API Client to call the LLM and get the response."""
    response = gemini_api_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction = INSTRUCTIONS,
            tools=[search_tool],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(
                disable=True
            ),
            tool_config=types.ToolConfig(
                function_calling_config=types.FunctionCallingConfig(mode="AUTO")
            ),
        )
    )

    try:
        if response.function_calls:
            print("Tool requested:", response.function_calls[0].name)
            print("Args:", response.function_calls[0].args)
        else:
            print("No tool call requested")

        print(response.function_calls[0])
        print(type(response.function_calls[0]))

    except Exception as e:
        print(f"Error occurred: {e}")

    return response.text


def ask_beta_2(question: str) -> str:
    """Answers a question by searching the FAQ database and using the LLM to generate a response."""
    search_results = search(question)
    prompt = build_prompt(question, search_results)
    answer = llm_gemini_2(prompt)
    return answer

In [25]:
print(ask_beta_2("Can I join ml ?"))

No tool call requested
Error occurred: 'NoneType' object is not subscriptable
I'm sorry, I can't answer this question. The provided context does not contain information about joining "ml".


In [45]:
print(ask_beta_2("I just found out about the course, can I still join?"))

Tool requested: search
Args: {'query': 'Can I still join?'}
id=None args={'query': 'Can I still join?'} name='search' partial_args=None will_continue=None
<class 'google.genai.types.FunctionCall'>
None
